# Day 20 練習：同一個旅行工具，Local MCP 和 Remote MCP 怎麼連？

這份練習只觀察一件事：**MCP Server 放在哪裡，以及 Agent 用什麼方式連到它。**

我們會做一個很小的「台南旅行工具」，它只會回傳範例天氣與行李提醒。它不讀取你的檔案、不使用 API Key、不連外，也沒有刪除、寄信或付款功能。旅行很刺激，但這份練習不用。

預計時間：約 20–30 分鐘。

## 你會看到什麼？

1. **Local MCP**：Python client 啟動自己電腦裡的 Server，透過 `stdio` 溝通。
2. **Remote MCP 的連線形式**：Python client 用網址連到 HTTP Server。為了安全，這台 HTTP Server 仍跑在 `127.0.0.1`；它是「像遠端一樣用網址連線」的實驗，不是假裝自己在雲端。
3. **真正 Remote MCP 的安全檢查**：最後可選填自己的網址，但預設只列出工具，完全不呼叫它們。

每一段都先列出工具，再呼叫一個只讀範例工具。你要觀察的是：工具的名稱和用途沒有變，變的是 Server 的位置與連線方式。

In [1]:
# 第一次執行時安裝 MCP Python SDK。已安裝的人不會重複安裝。
import asyncio
import importlib.metadata
import importlib.util
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

if importlib.util.find_spec("mcp") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mcp[cli]"])


print(f"Python：{sys.version.split()[0]}")
print(f"MCP Python SDK：{importlib.metadata.version('mcp')}")

Python：3.11.14
MCP Python SDK：2.0.0


## 1. 先準備同一份旅行工具

下面的 Server 有兩個 `Tools`：

- `get_trip_weather(city)`：回傳**虛構的範例天氣**，不會上網查真實天氣。
- `get_packing_tip(weather)`：依天氣提供行李提醒。

Local 與 HTTP 實驗都會使用這兩個工具。這樣比較時才公平：不是換了一個比較會講話的模型，而是把同一個櫃台放到不同位置。

In [2]:
SERVER_SOURCE = '''from mcp.server import MCPServer

mcp = MCPServer("台南旅行工具")

@mcp.tool()
def get_trip_weather(city: str) -> str:
    """查詢教學用的範例天氣；不會連到真實天氣服務。"""
    samples = {
        "台南": "範例資料：晴到多雲，午後可能有雨；記得帶傘，也記得帶人。",
        "高雄": "範例資料：晴朗炎熱；防曬的重要性高於自拍濾鏡。",
    }
    return samples.get(city, f"範例資料：目前沒有 {city} 的行程天氣。")

@mcp.tool()
def get_packing_tip(weather: str) -> str:
    """根據天氣描述提供教學用的行李提醒。"""
    if "雨" in weather:
        return "行李提醒：摺疊傘、薄外套、防水袋。手機淋雨不會自動變成防水版。"
    return "行李提醒：水壺、帽子與好走的鞋。旅程不是腳底按摩預約。"

if __name__ == "__main__":
    mcp.run()
'''

demo_dir = Path(tempfile.mkdtemp(prefix="day20_mcp_"))
stdio_server_path = demo_dir / "trip_mcp_stdio_server.py"
http_server_path = demo_dir / "trip_mcp_http_server.py"

stdio_server_path.write_text(SERVER_SOURCE, encoding="utf-8")
http_source = SERVER_SOURCE.replace(
    "mcp.run()",
    'mcp.run("streamable-http", host="127.0.0.1", port=8765)',
)
http_server_path.write_text(http_source, encoding="utf-8")

print(f"已在暫存資料夾準備 Server：{demo_dir.name}")
print("它會在最後一格被清理，不會讀取你的專案檔案。")

已在暫存資料夾準備 Server：day20_mcp_k0_deuj4
它會在最後一格被清理，不會讀取你的專案檔案。


## 2. Local MCP：由本機啟動，透過 `stdio` 對話

這裡的 client 會把 Server 當成子程序啟動。兩者透過標準輸入與輸出交換 MCP 訊息，因此不需要網址，也不需要開網路埠。

這種方式很適合「只服務自己這台電腦」的工具，例如讀取某個專案、呼叫本機開發工具。當然，能讀多少檔案仍取決於你給它什麼權限。

In [3]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

def show_result(result):
    """把 MCP 回傳內容用初學者看得懂的方式印出來。"""
    for item in getattr(result, "content", []):
        if hasattr(item, "text"):
            print(item.text)

async def run_local_demo():
    params = StdioServerParameters(
        command=sys.executable,
        args=[str(stdio_server_path)],
    )
    async with stdio_client(params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            result = await session.call_tool(
                "get_trip_weather", {"city": "台南"}
            )
            return tools.tools, result

local_tools, local_result = await run_local_demo()
print("Local MCP 看得到的工具：")
for tool in local_tools:
    print(f"- {tool.name}：{tool.description}")

print("\n呼叫 get_trip_weather('台南') 的結果：")
show_result(local_result)

Local MCP 看得到的工具：
- get_trip_weather：查詢教學用的範例天氣；不會連到真實天氣服務。
- get_packing_tip：根據天氣描述提供教學用的行李提醒。

呼叫 get_trip_weather('台南') 的結果：
範例資料：晴到多雲，午後可能有雨；記得帶傘，也記得帶人。


### 觀察

你剛剛看到的工具清單與回傳內容，都來自這台電腦啟動的 Server。這就是 Local MCP 的核心：**本機程式 + 本機啟動指令**。

它不是天然安全。若你把「讀取整個硬碟」這種工具交給它，Local 只代表資料沒有先送到遠端，不代表權限會突然變得有禮貌。

## 3. Remote MCP 的連線形式：用網址找到 Server

真正的 Remote MCP Server 會在另一台主機，client 透過網址連線。為了讓這份 Notebook 可以安全、完整地執行，我們先把 HTTP Server 放在 `127.0.0.1`。

因此，下一格示範的是 **HTTP 連線的形狀**，不是把你的資料送到雲端。之後若把網址換成公司服務或線上平台，才會變成真正的 Remote MCP；那時就必須開始問：誰能連？要不要登入？這個工具可以讀或改什麼？

In [4]:
from mcp.client.streamable_http import streamable_http_client

http_server = subprocess.Popen(
    [sys.executable, str(http_server_path)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
)
LOCAL_HTTP_URL = "http://127.0.0.1:8765/mcp"

async def run_http_demo():
    last_error = None
    for _ in range(20):
        try:
            async with streamable_http_client(LOCAL_HTTP_URL) as (read_stream, write_stream):
                async with ClientSession(read_stream, write_stream) as session:
                    await session.initialize()
                    tools = await session.list_tools()
                    result = await session.call_tool(
                        "get_packing_tip", {"weather": "午後有雨"}
                    )
                    return tools.tools, result
        except Exception as exc:
            last_error = exc
            await asyncio.sleep(0.25)
    raise RuntimeError(f"本機 HTTP MCP Server 未能啟動：{last_error}")

http_tools, http_result = await run_http_demo()
print(f"連線網址：{LOCAL_HTTP_URL}")
print("HTTP MCP 看得到的工具：")
for tool in http_tools:
    print(f"- {tool.name}：{tool.description}")

print("\n呼叫 get_packing_tip('午後有雨') 的結果：")
show_result(http_result)

連線網址：http://127.0.0.1:8765/mcp
HTTP MCP 看得到的工具：
- get_trip_weather：查詢教學用的範例天氣；不會連到真實天氣服務。
- get_packing_tip：根據天氣描述提供教學用的行李提醒。

呼叫 get_packing_tip('午後有雨') 的結果：
行李提醒：摺疊傘、薄外套、防水袋。手機淋雨不會自動變成防水版。


## 4. 對照：同一個工具，差別在哪裡？

| 問題 | Local MCP | Remote MCP |
| --- | --- | --- |
| Server 在哪裡？ | 你的電腦 | 另一台可經由網路找到的主機 |
| 怎麼連？ | 啟動本機程式，例如 `stdio` | 使用網址，例如 Streamable HTTP |
| 常見用途 | 專案檔案、開發工具、個人資料 | 公司服務、多人共用的平台、線上資料 |
| 優先確認什麼？ | 程式來源與可讀取的範圍 | 網址、登入方式、資料傳送與帳號權限 |

剛剛兩次列出的工具相同，因為 Server 的能力相同。MCP 的好處是 client 不必重學每一家服務的語言；但它不能替你判斷那把鑰匙該不該交出去。

## 5. 可選：檢查真正的 Remote MCP Server

只有在你**已確認 Server 來源、網址、登入方式與權限範圍**時，才填入自己的 MCP endpoint。這一格預設不會做任何事；填入後也只會初始化連線並列出工具，**不會呼叫工具**。

不要把 API Key 直接寫進 Notebook。若該 Server 需要登入或 Token，請依它的官方文件設定安全的認證流程。

In [5]:
REMOTE_MCP_URL = ""  # 例如：https://你的服務.example/mcp

async def inspect_remote_server(url):
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            return tools.tools

if not REMOTE_MCP_URL.strip():
    print("安全跳過：尚未填入 Remote MCP URL。")
else:
    remote_tools = await inspect_remote_server(REMOTE_MCP_URL.strip())
    print("已連線；以下只列出工具，沒有呼叫任何工具：")
    for tool in remote_tools:
        print(f"- {tool.name}：{tool.description}")

安全跳過：尚未填入 Remote MCP URL。


## 6. 帶走三個判斷

1. 先問 Server 在本機還是遠端，再決定要看檔案範圍或網路與帳號權限。
2. 先 `list_tools()` 看它能做什麼，再考慮是否真的要呼叫；不要因為看到「超強助手」四個字就直接按下同意。
3. 敏感操作應採最小權限，並保留人類確認。工作規則可以提醒 Agent，但真正的門禁仍要靠權限與確認步驟。

這就是 Day 20 最重要的一句話：**MCP 讓工具比較容易接上，不會自動讓權限變得安全。**

In [6]:
# 清理本機 HTTP Server 與暫存檔。整份 Notebook 跑完後執行這一格。
if "http_server" in globals() and http_server.poll() is None:
    http_server.terminate()
    try:
        http_server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        http_server.kill()
    print("本機 HTTP MCP Server 已停止。")

if "demo_dir" in globals() and demo_dir.exists():
    shutil.rmtree(demo_dir)
    print("教學用暫存檔已清理。")

本機 HTTP MCP Server 已停止。
教學用暫存檔已清理。
